# Blind Test with Cold Start Setting Dataset(s) Preprocessing #

DeepRSMA have curated blind test evaluation dataset splits (5 fold) with cold started setting for blind RNA, blind small molecule, and for both blind RNA and small molecule.

In that notebook, we'll analyse their splits and make sure its valid. Moreover, we'll generate "inner cold start" splits for cross-validation with early stopping in order to decide the number of epochs we'll train each one of the five folds before testing on the corresponding fold test split.

In [63]:
import os

import pandas as pd
import numpy as np

In [32]:
COLD_PATH = '/home/gavrilev/Thesis/src/datasets/blind_test/cold_' # path to cold start dataset(s) directory
ALL_SF_PATH = '/home/gavrilev/Thesis/src/datasets/RSIM/All_sf_dataset_v1.csv'

In [33]:
all_sf_df = pd.read_csv(ALL_SF_PATH, delimiter='\t')
all_sf_df.head()

,Entry_ID,SMILES,Target_RNA_sequence,Molecule_name,Molecule_ID,Target_RNA_name,Target_RNA_ID,pKd
0,2091,N[C@@H]1C[C@H](N)[C@H]([C@H]([C@H]1OCCCc1nnn(c...,CGCGCAGUCAGUAGCGCG,B-13,Target_lig_1138,RNA hairpin loop IV,Target_381,6.387216
1,162,C1CN(CCC1C(=O)NCCN2CCOCC2)CC3=CC=C(C=C3)Cl,GAGCGUCACACCUUCGGGUGAAGUCGCUC,1-(4-chlorobenzyl)-N-(2-morpholinoethyl)piperi...,Target_lig_132,A-site 16S rRNA_E_coli (1),Target_3,3.602060
2,270,c1c(ccc2Sc3ccccc3N(c12)CCCNCN(CO)CO)C(=O)C,GGCAGAUCUGAGCCUGGGAGCUCUCUGCC,3_phenothiazine_analog_1,Target_lig_210,HIV-1 TAR RNA,Target_19,2.522879
3,1327,C(CN1C=C(CNC2=C3C=CC=CC3=[NH+]C3=C2OC2=C3C=CC=...,GGGUUAGGGU,Quindoline CK1-14,Target_lig_723,TRF2:TERRA G-quadruplex (TERRA22):Telomeric du...,Target_262,6.272784
4,1907,C1=CC=C(C=C1)C2=NN=C(O2)C3=NC4=CC=CC=C4C(=N3)N...,UGUGGGAGGGGCGGGUCUGGG,NRAS SAR 7,Target_lig_1057,NRAS G4 motif,Target_259,6.443697


## Helper Methods for Preprocessing ##

In [50]:
def get_all_cold_type_concat(cold_type: str = 'rna') -> pd.DataFrame():
    if cold_type not in ['rna', 'mol', 'rm']:
        raise ValueError("must be on of the next options -> ['rna', 'mol', 'rm']")
        return
    df1 = pd.read_csv(COLD_PATH + cold_type +'1.csv', delimiter=',')
    df2 = pd.read_csv(COLD_PATH + cold_type +'2.csv', delimiter=',')
    df3 = pd.read_csv(COLD_PATH + cold_type +'3.csv', delimiter=',')
    df4 = pd.read_csv(COLD_PATH + cold_type +'4.csv', delimiter=',')
    df5 = pd.read_csv(COLD_PATH + cold_type +'5.csv', delimiter=',')
    dfs = [df1, df2, df3, df4, df5]
    for i,d in enumerate(dfs):
        print(80*"#" + "\n" + f"Fold number -> {i+1}, shape -> {d.shape}")
    df_cold_all = pd.concat([df1, df2,df3, df4,df5], axis=0).reset_index()
    print(80*"#" + "\n" + f'Cold type -> {cold_type}, concat shape {df_cold_all.shape}')

    return df_cold_all, dfs

In [61]:
def _infer_col(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None


def infer_dataset_columns(all_df: pd.DataFrame):
    entry_col = _infer_col(all_df, ["Entry_ID", "entry_id", "ENTRY_ID"])
    if entry_col is None:
        raise ValueError("Could not find Entry_ID column in all_df")

    # try common RNA identifier columns
    rna_col = _infer_col(all_df, [
        "RNA_ID", "rna_id", "RNA", "rna", "RNA_seq", "rna_seq", "Target_RNA_sequence",
        "RNA_sequence", "rna_sequence", "Sequence", "sequence", "Target_RNA"
    ])

    # try common molecule columns
    mol_col = _infer_col(all_df, [
        "SMILES", "smiles", "Ligand_SMILES", "ligand_smiles",
        "Drug_SMILES", "drug_smiles", "Molecule", "molecule"
    ])

    return entry_col, rna_col, mol_col

In [52]:
def build_outer_train_test(all_df: pd.DataFrame, outer_dfs, outer_i: int, entry_col_all: str):
    # DeepRSMA fold CSVs typically have Entry_ID
    entry_col_fold = _infer_col(outer_dfs[outer_i], ["Entry_ID", "entry_id", "ENTRY_ID"])
    if entry_col_fold is None:
        raise ValueError("Could not find Entry_ID column in outer fold csv")

    test_entry_ids = outer_dfs[outer_i][entry_col_fold].tolist()

    train_entry_ids = []
    for j, df_j in enumerate(outer_dfs):
        if j != outer_i:
            entry_col_fold_j = _infer_col(df_j, ["Entry_ID", "entry_id", "ENTRY_ID"])
            train_entry_ids.extend(df_j[entry_col_fold_j].tolist())

    test_df = all_df[all_df[entry_col_all].isin(test_entry_ids)].reset_index(drop=True)
    train_df = all_df[all_df[entry_col_all].isin(train_entry_ids)].reset_index(drop=True)

    return train_df, test_df

In [3]:
for i in range(1,6):
    print(i)

1
2
3
4
5


In [53]:
def make_inner_group_folds(train_df: pd.DataFrame, group_col: str, n_splits: int = 5, seed: int = 2):
    """
    Create inner validation folds for a group-wise (cold-start) split.

    Unique values of `group_col` are shuffled (controlled by `seed`) and partitioned into
    `n_splits` disjoint subsets. For each subset, the corresponding validation fold contains
    all rows whose group is in that subset.

    Returns a list of length `n_splits`, where each element is a validation DataFrame.
    """
    groups = train_df[group_col].dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    group_folds = np.array_split(groups, n_splits)

    inner_val_dfs = []
    for val_groups in group_folds:
        val_df = train_df[train_df[group_col].isin(val_groups)].reset_index(drop=True)
        inner_val_dfs.append(val_df)

    return inner_val_dfs

In [77]:
# =============================================================================
# Seed search scoring utilities
# =============================================================================
def _hist_distance_to_overall(y_overall: np.ndarray, y_fold: np.ndarray, bins: np.ndarray) -> float:
    """
    Compute mean absolute difference between two *density* histograms.

    A density histogram normalizes bin counts such that the integral over bins is ~1,
    making histograms comparable across different fold sizes.

    Parameters
    ----------
    y_overall : np.ndarray
        Target values for the overall dataset (all inner folds combined).

    y_fold : np.ndarray
        Target values for a single fold.

    bins : np.ndarray
        Histogram bin edges.

    Returns
    -------
    float
        Mean absolute difference between histogram densities (L1 distance averaged over bins).
        Lower values indicate a fold target distribution closer to the overall distribution.
    """
    h_all, _ = np.histogram(y_overall, bins=bins, density=True)
    h_f, _ = np.histogram(y_fold, bins=bins, density=True)
    return float(np.mean(np.abs(h_f - h_all)))


def _quantile_profile(y: np.ndarray, qs=(0.05, 0.25, 0.50, 0.75, 0.95)) -> np.ndarray:
    """
    Compute a vector of quantiles of a 1D target array.

    Parameters
    ----------
    y : np.ndarray
        Target values.

    qs : tuple, default=(0.05, 0.25, 0.50, 0.75, 0.95)
        Quantile levels to compute.

    Returns
    -------
    np.ndarray
        Quantile values at the specified levels.
    """
    return np.quantile(y, qs)


def score_group_folds_regression(
    folds: list[pd.DataFrame],
    y_col: str,
    group_col: str,
    bins_count: int = 25,
    quantiles=(0.05, 0.25, 0.50, 0.75, 0.95),
    w_rows: float = 1.0,
    w_groups: float = 0.5,
    w_hist: float = 1.0,
    w_meanstd: float = 0.5,
    w_quant: float = 0.5,
) -> dict:
    """
    Compute a scalar imbalance score for a set of group-wise folds in a regression setting.

    The score is a weighted sum of penalties capturing:
      - Row-count imbalance across folds (rows_cv)
      - Group-count imbalance across folds (groups_cv)
      - Label distribution mismatch (histogram distance, mean/std spread, quantile mismatch)

    Lower score indicates more balanced folds.

    Definitions
    -----------
    - rows_cv:   std(rows_per_fold) / mean(rows_per_fold)
    - groups_cv: std(groups_per_fold) / mean(groups_per_fold)

    - hist_score: average L1 distance between each fold's density histogram and the overall
      density histogram.

    - meanstd_score: normalized spread of fold means and fold standard deviations:
        std(fold_means)/std(overall) + std(fold_stds)/std(overall)

    - quant_score: normalized average L1 distance between each fold's quantile profile and the
      overall quantile profile.

    Parameters
    ----------
    folds : list[pd.DataFrame]
        List of fold DataFrames (typically inner validation folds).

    y_col : str
        Column name of the regression target.

    group_col : str
        Column name used for groups (must exist in each fold).

    bins_count : int, default=25
        Number of histogram bins used for the histogram-based distance.

    quantiles : tuple, default=(0.05, 0.25, 0.50, 0.75, 0.95)
        Quantile levels used for the quantile-profile distance.

    w_rows, w_groups, w_hist, w_meanstd, w_quant : float
        Weights for each component in the final score.

    Returns
    -------
    dict
        Dictionary containing the total score and individual component values, plus diagnostic
        fold sizes:
          - score, rows_cv, groups_cv, hist_score, meanstd_score, quant_score
          - row_counts, group_counts
    """
    y_all = np.concatenate([f[y_col].astype(float).to_numpy() for f in folds])
    if y_all.size == 0:
        raise ValueError("Empty y across folds")

    y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
    if y_min == y_max:
        bins = np.array([y_min, y_max + 1e-6])
    else:
        bins = np.linspace(y_min, y_max, bins_count + 1)

    row_counts = np.array([len(f) for f in folds], dtype=float)
    grp_counts = np.array([f[group_col].nunique() for f in folds], dtype=float)

    def cv(x):
        m = float(np.mean(x))
        return float(np.std(x) / (m + 1e-12))

    rows_cv = cv(row_counts)
    groups_cv = cv(grp_counts)

    hist_dists = []
    fold_means = []
    fold_stds = []
    quant_dists = []

    q_all = _quantile_profile(y_all, qs=quantiles)

    for f in folds:
        y_f = f[y_col].astype(float).to_numpy()
        if y_f.size == 0:
            # Empty fold indicates an invalid split; apply a large penalty.
            return {"score": 1e9}

        hist_dists.append(_hist_distance_to_overall(y_all, y_f, bins))
        fold_means.append(float(np.mean(y_f)))
        fold_stds.append(float(np.std(y_f)))

        q_f = _quantile_profile(y_f, qs=quantiles)
        quant_dists.append(float(np.mean(np.abs(q_f - q_all))))

    hist_score = float(np.mean(hist_dists))

    overall_std = float(np.std(y_all)) + 1e-12
    mean_spread = float(np.std(fold_means) / overall_std)
    std_spread  = float(np.std(fold_stds) / overall_std)
    meanstd_score = mean_spread + std_spread

    quant_score = float(np.mean(quant_dists) / (overall_std + 1e-12))

    total = (
        w_rows * rows_cv +
        w_groups * groups_cv +
        w_hist * hist_score +
        w_meanstd * meanstd_score +
        w_quant * quant_score
    )

    return {
        "score": float(total),
        "rows_cv": rows_cv,
        "groups_cv": groups_cv,
        "hist_score": hist_score,
        "meanstd_score": meanstd_score,
        "quant_score": quant_score,
        "row_counts": row_counts.astype(int).tolist(),
        "group_counts": grp_counts.astype(int).tolist(),
    }


def find_best_seed_for_inner_folds(
    train_df: pd.DataFrame,
    group_col: str,
    y_col: str,
    n_splits: int = 5,
    seeds=range(0, 200),
    bins_count: int = 25,
    quantiles=(0.05, 0.25, 0.50, 0.75, 0.95),
    weights=None,
) -> tuple[int, pd.DataFrame]:
    """
    Search over multiple random seeds to select the most balanced group-wise inner split.

    For each seed, group-wise folds are built using `make_inner_group_folds`, then scored
    using `score_group_folds_regression`. The seed minimizing the score is selected.

    Parameters
    ----------
    train_df : pd.DataFrame
        Outer-training dataset to be split into inner folds.

    group_col : str
        Grouping column name (e.g., RNA identifier for cold-RNA).

    y_col : str
        Regression target column name.

    n_splits : int, default=5
        Number of inner folds.

    seeds : iterable, default=range(0, 200)
        Candidate seeds to evaluate.

    bins_count : int, default=25
        Number of histogram bins for histogram-based distance.

    quantiles : tuple, default=(0.05, 0.25, 0.50, 0.75, 0.95)
        Quantiles used for quantile-profile distance.

    weights : dict or None
        Optional component weights. If None, default weights are used:
          w_rows=1.0, w_groups=0.5, w_hist=1.0, w_meanstd=0.5, w_quant=0.5

    Returns
    -------
    best_seed : int
        Seed achieving the minimum score.

    results_df : pd.DataFrame
        DataFrame containing scores and diagnostic components per seed, sorted by score ascending.
        Columns include: seed, score, rows_cv, groups_cv, hist_score, meanstd_score, quant_score,
        row_counts, group_counts.
    """
    if weights is None:
        weights = dict(w_rows=1.0, w_groups=0.5, w_hist=1.0, w_meanstd=0.5, w_quant=0.5)

    rows = []
    for s in seeds:
        folds = make_inner_group_folds(train_df, group_col=group_col, n_splits=n_splits, seed=int(s))
        stats = score_group_folds_regression(
            folds,
            y_col=y_col,
            group_col=group_col,
            bins_count=bins_count,
            quantiles=quantiles,
            **weights
        )
        rows.append({
            "seed": int(s),
            "score": stats["score"],
            "rows_cv": stats["rows_cv"],
            "groups_cv": stats["groups_cv"],
            "hist_score": stats["hist_score"],
            "meanstd_score": stats["meanstd_score"],
            "quant_score": stats["quant_score"],
            "row_counts": stats["row_counts"],
            "group_counts": stats["group_counts"],
        })

    results_df = pd.DataFrame(rows).sort_values("score", ascending=True).reset_index(drop=True)
    best_seed = int(results_df.loc[0, "seed"])
    return best_seed, results_df

In [78]:
# =============================================================================
# Inner-fold construction
# =============================================================================
def make_inner_group_folds(
    train_df: pd.DataFrame,
    group_col: str,
    n_splits: int = 5,
    seed: int = 2
):
    """
    Construct inner validation folds for group-wise (cold-start) cross-validation.

    A group-wise split assigns entire groups (defined by `group_col`) to folds, rather than
    individual rows. This enforces a cold-start constraint: groups appearing in a validation
    fold do not appear in the corresponding inner training set.

    Method
    ------
    1) Extract unique group identifiers from `train_df[group_col]`.
    2) Shuffle these identifiers using `seed` for reproducibility.
    3) Partition the shuffled identifiers into `n_splits` disjoint subsets.
    4) For each subset, create a validation DataFrame containing all rows whose group is
       in that subset.

    Parameters
    ----------
    train_df : pd.DataFrame
        Outer-training dataset (e.g., the file saved as train.csv for an outer fold).

    group_col : str
        Column name defining the grouping key (e.g., RNA identifier for cold-RNA,
        SMILES or molecule identifier for cold-molecule).

    n_splits : int, default=5
        Number of inner folds to create.

    seed : int, default=2
        Random seed controlling the shuffling of group identifiers.

    Returns
    -------
    inner_val_dfs : list[pd.DataFrame]
        List of length `n_splits`. Element k is the validation fold DataFrame for inner fold k.

    Notes
    -----
    - This function returns *validation folds only*. The inner training set for fold k is
      derived as `train_df` minus `inner_val_dfs[k]` (e.g., by Entry_ID).
    - Fold sizes in terms of number of rows can vary substantially because different groups
      may contribute different numbers of rows.
    """
    groups = train_df[group_col].dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    group_folds = np.array_split(groups, n_splits)

    inner_val_dfs = []
    for val_groups in group_folds:
        val_df = train_df[train_df[group_col].isin(val_groups)].reset_index(drop=True)
        inner_val_dfs.append(val_df)

    return inner_val_dfs

In [83]:
# =============================================================================
# Main preprocessing with seed search
# =============================================================================
def preprocess_deeprsma_cold_inner_cv_with_seed_search(
    cold_type: str = "rna",
    all_df: pd.DataFrame = None,
    out_root: str = "/home/gavrilev/Thesis/src/datasets/blind_test/processed",
    n_inner: int = 5,
    y_col: str = "pKd",
    seeds_to_try=range(0, 10000),
    score_bins_count: int = 25,
    score_quantiles=(0.05, 0.25, 0.50, 0.75, 0.95),
    score_weights=None,
):
    """
    Generate outer cold-start train/test splits (given DeepRSMA blind-test folds), then create
    inner group-wise folds for early stopping selection, using a seed-search strategy to
    improve balance.

    For each outer fold:
      1) Construct outer train/test splits from the provided DeepRSMA fold CSVs.
      2) Verify cold-start constraint: no group overlap between outer train and outer test.
      3) Save `train.csv` and `test.csv` under the outer fold directory.
      4) Perform seed search on outer-train to choose an inner-fold seed that minimizes an
         imbalance score (rows + label distribution).
      5) Save:
           - `seed_search_results.csv` (all evaluated seeds and their scores)
           - `best_seed.txt` (chosen seed)
           - `inner_fold_1..n_inner.csv` (inner validation folds built with best seed)
      6) Verify cold-start constraint within inner CV: no group overlap between each inner
         training set (train minus inner fold) and its validation fold.

    Parameters
    ----------
    cold_type : str, default="rna"
        Cold-start mode. Supported: "rna" (group_col inferred as RNA identifier),
        "mol" (group_col inferred as molecule identifier). "rm" is not implemented here.

    all_df : pd.DataFrame
        Full dataset DataFrame containing at least:
          - Entry_ID column (or equivalent)
          - group column (RNA or molecule identifier)
          - regression target column `y_col`

    out_root : str
        Output root directory where processed splits will be written.

    n_inner : int, default=5
        Number of inner folds to create.

    y_col : str, default="pKd_norm"
        Regression target column name used for scoring label distribution balance.

    seeds_to_try : iterable, default=range(0, 500)
        Candidate seeds to evaluate for choosing the best inner fold split per outer fold.

    score_bins_count : int, default=25
        Number of bins used in histogram-based label distribution distance.

    score_quantiles : tuple
        Quantile levels used to compare fold quantile profiles to the overall distribution.

    score_weights : dict or None
        Optional weights passed to the scoring function. If None, defaults are used.

    Returns
    -------
    None
        Files are written to disk under:
          {out_root}/cold_{cold_type}/outer_fold_{i}/
    """
    if all_df is None:
        raise ValueError("all_df is None")

    # External helpers assumed to exist from previous code:
    # - get_all_cold_type_concat
    # - infer_dataset_columns
    # - build_outer_train_test

    _, outer_dfs = get_all_cold_type_concat(cold_type)
    entry_col, rna_col, mol_col = 'Entry_ID' ,'SMILES', 'Target_RNA_sequence'

    if cold_type == "rna":
        if rna_col is None:
            raise ValueError("Could not infer RNA identifier column in all_df for cold_type='rna'")
        group_col = rna_col
    elif cold_type == "mol":
        if mol_col is None:
            raise ValueError("Could not infer molecule identifier column in all_df for cold_type='mol'")
        group_col = mol_col
    else:
        raise ValueError("cold_type='rm' inner-CV not implemented here (do rna/mol first)")

    if y_col not in all_df.columns:
        raise ValueError(f"y_col='{y_col}' not found in all_df columns")

    base_dir = os.path.join(out_root, f"cold_{cold_type}")
    os.makedirs(base_dir, exist_ok=True)

    for i in range(5):
        outer_dir = os.path.join(base_dir, f"outer_fold_{i+1}")
        os.makedirs(outer_dir, exist_ok=True)

        train_df, test_df = build_outer_train_test(all_df, outer_dfs, i, entry_col)
        print(f'Fold {i+1} train & test shapes -> {train_df.shape}, {test_df.shape}')

        # Outer cold-start check (no group overlap between train and test)
        overlap_outer = set(train_df[group_col].unique()).intersection(set(test_df[group_col].unique()))
        print(80 * "#")
        print(f"[cold_{cold_type}] Outer fold {i+1}: train={len(train_df)}, test={len(test_df)}, "
              f"unique_train_groups={train_df[group_col].nunique()}, unique_test_groups={test_df[group_col].nunique()}, "
              f"overlap_groups={len(overlap_outer)}")
        assert len(overlap_outer) == 0

        # Save outer splits
        train_df.to_csv(os.path.join(outer_dir, "train.csv"), index=False)
        test_df.to_csv(os.path.join(outer_dir, "test.csv"), index=False)

        # Seed search on outer-train to select best inner split
        best_seed, results_df = find_best_seed_for_inner_folds(
            train_df=train_df,
            group_col=group_col,
            y_col=y_col,
            n_splits=n_inner,
            seeds=seeds_to_try,
            bins_count=score_bins_count,
            quantiles=score_quantiles,
            weights=score_weights,
        )

        # Save seed-search artifacts
        results_df.to_csv(os.path.join(outer_dir, "seed_search_results.csv"), index=False)
        with open(os.path.join(outer_dir, "best_seed.txt"), "w") as f:
            f.write(str(best_seed) + "\n")

        top = results_df.head(5)[["seed", "score", "rows_cv", "groups_cv", "hist_score", "meanstd_score", "quant_score", "row_counts"]]
        print(f"[cold_{cold_type}] Outer fold {i+1}: best_seed={best_seed}")
        print(top.to_string(index=False))

        # Build inner folds using the chosen seed and save them
        inner_val_dfs = make_inner_group_folds(train_df, group_col=group_col, n_splits=n_inner, seed=best_seed)

        all_val_entry_ids = []
        for k, val_df in enumerate(inner_val_dfs):
            inner_val_path = os.path.join(outer_dir, f"inner_fold_{k+1}.csv")
            val_df.to_csv(inner_val_path, index=False)

            # Inner training set is defined as outer-train minus the validation fold.
            inner_train_df = train_df[~train_df[entry_col].isin(val_df[entry_col])]

            # Inner cold-start check (no group overlap between inner-train and inner-val)
            overlap_inner = set(inner_train_df[group_col].unique()).intersection(set(val_df[group_col].unique()))
            assert len(overlap_inner) == 0

            all_val_entry_ids.extend(val_df[entry_col].tolist())

            print(f"  inner_fold {k+1}: val={len(val_df)}, inner_train={len(inner_train_df)}, "
                  f"val_unique_groups={val_df[group_col].nunique()}, overlap_groups={len(overlap_inner)}")

        # Ensure inner validation folds partition the outer-train by Entry_ID
        assert len(all_val_entry_ids) == len(train_df)
        assert len(set(all_val_entry_ids)) == len(train_df)

    print("\nDONE. Created outer_fold_1..5 directories with train/test + inner_fold_1..5.csv "
          "chosen by seed-search per outer fold.")

## Preprocess RNA Cold Type ##

In [56]:
df_cold_all, all_dfs = get_all_cold_type_concat(cold_type='rna')

################################################################################
Fold number -> 1, shape -> (183, 8)
################################################################################
Fold number -> 2, shape -> (243, 8)
################################################################################
Fold number -> 3, shape -> (305, 8)
################################################################################
Fold number -> 4, shape -> (479, 8)
################################################################################
Fold number -> 5, shape -> (229, 8)
################################################################################
Cold type -> rna, concat shape (1439, 9)


In [57]:
df_cold_all.shape

(1439, 9)

In [60]:
df_cold_all.head()

,index,Entry_ID,SMILES,Target_RNA_sequence,Molecule_name,Molecule_ID,Target_RNA_name,Target_RNA_ID,pKd
0,0,39,C1=CC=C2C(=C1)C(=O)C3=C(C2=O)C(=C(C=C3NC4=CC(=...,GGGAGAATTCCCGCGGCAGAAGCCCACCTGGCTTTGAACTCTATGT...,Cibacron blue,Target_lig_27,RNA_APTAMER_CB_42,Target_4,4.000000
1,1,93,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FAD,Target_lig_67,35 nucleotide RNA,Target_26,4.638272
2,2,94,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,FMN,Target_lig_68,35 nucleotide RNA,Target_26,6.301030
3,3,95,C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)CO)O)O)N,GGCGUGUAGGAUAUGCUUCGGCAGAAGGACACGCC,ADENOSINE,Target_lig_69,35 nucleotide RNA,Target_26,2.397940
4,4,96,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,GGAACGAGGGAUGGAGGAGGAGUCGUUCC,riboflavin,Target_lig_70,Riboflavin binding RNA aptamer,Target_29,-0.477121


In [84]:
preprocess_deeprsma_cold_inner_cv_with_seed_search(cold_type="rna",
    all_df=all_sf_df)

################################################################################
Fold number -> 1, shape -> (183, 8)
################################################################################
Fold number -> 2, shape -> (243, 8)
################################################################################
Fold number -> 3, shape -> (305, 8)
################################################################################
Fold number -> 4, shape -> (479, 8)
################################################################################
Fold number -> 5, shape -> (229, 8)
################################################################################
Cold type -> rna, concat shape (1439, 9)
Fold 1 train & test shapes -> (1256, 8), (183, 8)
################################################################################
[cold_rna] Outer fold 1: train=1256, test=183, unique_train_groups=236, unique_test_groups=59, overlap_groups=0
[cold_rna] Outer fold 1: best_seed=8534
 seed    s

In [65]:
import os
import pandas as pd

def reg_stats(df, y_col, group_col=None):
    y = df[y_col].astype(float)
    out = {
        "rows": len(df),
        "y_mean": y.mean(),
        "y_std": y.std(),
        "y_min": y.min(),
        "y_q05": y.quantile(0.05),
        "y_q50": y.quantile(0.50),
        "y_q95": y.quantile(0.95),
        "y_max": y.max(),
    }
    if group_col is not None:
        cnt = df.groupby(group_col).size()
        out.update({
            "unique_groups": df[group_col].nunique(),
            "rows_per_group_min": int(cnt.min()),
            "rows_per_group_med": float(cnt.median()),
            "rows_per_group_max": int(cnt.max()),
        })
    return out

def analyze_saved_splits(base_dir, y_col, rna_col, entry_col="Entry_ID"):
    for i in range(1, 6):
        outer_dir = os.path.join(base_dir, f"outer_fold_{i}")
        train_path = os.path.join(outer_dir, "train.csv")
        test_path  = os.path.join(outer_dir, "test.csv")

        train_df = pd.read_csv(train_path)
        test_df  = pd.read_csv(test_path)

        # duplicates sanity
        assert train_df[entry_col].is_unique
        assert test_df[entry_col].is_unique
        assert set(train_df[entry_col]).isdisjoint(set(test_df[entry_col]))

        print("="*90)
        print(f"OUTER {i}")
        print("train:", reg_stats(train_df, y_col, group_col=rna_col))
        print("test :", reg_stats(test_df,  y_col, group_col=rna_col))

        for k in range(1, 6):
            val_path = os.path.join(outer_dir, f"inner_fold_{k}.csv")
            val_df = pd.read_csv(val_path)
            inner_train_df = train_df[~train_df[entry_col].isin(val_df[entry_col])]

            # cold-start sanity inside outer-train
            assert set(inner_train_df[rna_col].unique()).isdisjoint(set(val_df[rna_col].unique()))

            print(f"  inner {k} val :", reg_stats(val_df, y_col, group_col=rna_col))
            print(f"  inner {k} trn :", reg_stats(inner_train_df, y_col, group_col=rna_col))




In [66]:
# example:
analyze_saved_splits(
   base_dir="/home/gavrilev/Thesis/src/datasets/blind_test/processed/cold_rna",
   y_col="pKd",
   rna_col="Target_RNA_sequence"     
 )

OUTER 1
train: {'rows': 1256, 'y_mean': 5.348035953353907, 'y_std': 1.4040246313343334, 'y_min': 0.154901959985743, 'y_q05': 3.0, 'y_q50': 5.397940008672039, 'y_q95': 7.526559559485515, 'y_max': 11.301029995664, 'unique_groups': 236, 'rows_per_group_min': 1, 'rows_per_group_med': 1.0, 'rows_per_group_max': 242}
test : {'rows': 183, 'y_mean': 5.677830511874489, 'y_std': 1.3137074418223293, 'y_min': -0.477121254719662, 'y_q05': 3.167473766080085, 'y_q50': 5.62160209905186, 'y_q95': 7.871071463649144, 'y_max': 9.0, 'unique_groups': 59, 'rows_per_group_min': 1, 'rows_per_group_med': 2.0, 'rows_per_group_max': 20}
  inner 1 val : {'rows': 223, 'y_mean': 5.887731350508981, 'y_std': 1.1691182900529227, 'y_min': 2.744727494896689, 'y_q05': 4.051972281235564, 'y_q50': 5.69897000433602, 'y_q95': 8.111042586470768, 'y_max': 9.0, 'unique_groups': 48, 'rows_per_group_min': 1, 'rows_per_group_med': 2.0, 'rows_per_group_max': 26}
  inner 1 trn : {'rows': 1033, 'y_mean': 5.231528621731854, 'y_std': 1.

In [67]:
import numpy as np
import pandas as pd


def make_inner_group_folds(train_df: pd.DataFrame, group_col: str, n_splits: int = 5, seed: int = 2):
    """
    Create inner validation folds for a group-wise (cold-start) split.

    Unique values of `group_col` are shuffled (controlled by `seed`) and partitioned into
    `n_splits` disjoint subsets. For each subset, the corresponding validation fold contains
    all rows whose group is in that subset.

    Returns a list of length `n_splits`, where each element is a validation DataFrame.
    """
    groups = train_df[group_col].dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(groups)
    group_folds = np.array_split(groups, n_splits)

    inner_val_dfs = []
    for val_groups in group_folds:
        val_df = train_df[train_df[group_col].isin(val_groups)].reset_index(drop=True)
        inner_val_dfs.append(val_df)

    return inner_val_dfs


def _hist_distance_to_overall(y_overall: np.ndarray, y_fold: np.ndarray, bins: np.ndarray) -> float:
    """Mean absolute difference between density histograms (L1 distance)."""
    # density=True makes area sum ~1, comparable across different fold sizes
    h_all, _ = np.histogram(y_overall, bins=bins, density=True)
    h_f, _ = np.histogram(y_fold, bins=bins, density=True)
    return float(np.mean(np.abs(h_f - h_all)))


def _quantile_profile(y: np.ndarray, qs=(0.05, 0.25, 0.50, 0.75, 0.95)) -> np.ndarray:
    return np.quantile(y, qs)


def score_group_folds_regression(
    folds: list[pd.DataFrame],
    y_col: str,
    group_col: str,
    bins_count: int = 25,
    quantiles=(0.05, 0.25, 0.50, 0.75, 0.95),
    w_rows: float = 1.0,
    w_groups: float = 0.5,
    w_hist: float = 1.0,
    w_meanstd: float = 0.5,
    w_quant: float = 0.5,
) -> dict:
    """
    Compute a scalar score for how "balanced" a set of folds is for regression.

    Lower score is better.

    Components:
      - rows balance: coefficient of variation of row counts across folds
      - group balance: coefficient of variation of unique group counts across folds
      - label histogram: average L1 distance of fold density histogram vs overall
      - mean/std spread: std of fold means + std of fold stds (normalized)
      - quantile spread: average L1 distance of fold quantile profile vs overall
    """
    # Collect y
    y_all = np.concatenate([f[y_col].astype(float).to_numpy() for f in folds])
    if y_all.size == 0:
        raise ValueError("Empty y across folds")

    # Bins based on overall y range
    y_min, y_max = float(np.min(y_all)), float(np.max(y_all))
    if y_min == y_max:
        # Degenerate target; any fold split is equivalent in label distribution
        bins = np.array([y_min, y_max + 1e-6])
    else:
        bins = np.linspace(y_min, y_max, bins_count + 1)

    # Row counts + group counts
    row_counts = np.array([len(f) for f in folds], dtype=float)
    grp_counts = np.array([f[group_col].nunique() for f in folds], dtype=float)

    def cv(x):
        m = float(np.mean(x))
        return float(np.std(x) / (m + 1e-12))

    rows_cv = cv(row_counts)
    groups_cv = cv(grp_counts)

    # Label histogram distance
    hist_dists = []
    fold_means = []
    fold_stds = []
    quant_dists = []

    q_all = _quantile_profile(y_all, qs=quantiles)

    for f in folds:
        y_f = f[y_col].astype(float).to_numpy()
        if y_f.size == 0:
            # hard penalty for empty fold
            return {"score": 1e9}

        hist_dists.append(_hist_distance_to_overall(y_all, y_f, bins))
        fold_means.append(float(np.mean(y_f)))
        fold_stds.append(float(np.std(y_f)))
        q_f = _quantile_profile(y_f, qs=quantiles)
        quant_dists.append(float(np.mean(np.abs(q_f - q_all))))

    hist_score = float(np.mean(hist_dists))

    # Normalize mean/std spread by overall std so score scale is reasonable
    overall_std = float(np.std(y_all)) + 1e-12
    mean_spread = float(np.std(fold_means) / overall_std)
    std_spread  = float(np.std(fold_stds) / overall_std)
    meanstd_score = mean_spread + std_spread

    quant_score = float(np.mean(quant_dists) / (overall_std + 1e-12))

    # Weighted total score
    total = (
        w_rows * rows_cv +
        w_groups * groups_cv +
        w_hist * hist_score +
        w_meanstd * meanstd_score +
        w_quant * quant_score
    )

    return {
        "score": float(total),
        "rows_cv": rows_cv,
        "groups_cv": groups_cv,
        "hist_score": hist_score,
        "meanstd_score": meanstd_score,
        "quant_score": quant_score,
        "row_counts": row_counts.astype(int).tolist(),
        "group_counts": grp_counts.astype(int).tolist(),
        "fold_means": fold_means,
        "fold_stds": fold_stds,
    }


def find_best_seed_for_inner_folds(
    train_df: pd.DataFrame,
    group_col: str,
    y_col: str,
    n_splits: int = 5,
    seeds=range(0, 200),
    bins_count: int = 25,
    quantiles=(0.05, 0.25, 0.50, 0.75, 0.95),
    weights=None,
    top_k: int = 10,
) -> tuple[int, pd.DataFrame]:
    """
    Try multiple seeds and return:
      - best_seed (lowest score)
      - results_df (sorted), with diagnostic components per seed
    """
    if weights is None:
        weights = dict(w_rows=1.0, w_groups=0.5, w_hist=1.0, w_meanstd=0.5, w_quant=0.5)

    rows = []
    for s in seeds:
        folds = make_inner_group_folds(train_df, group_col=group_col, n_splits=n_splits, seed=int(s))
        stats = score_group_folds_regression(
            folds, y_col=y_col, group_col=group_col,
            bins_count=bins_count, quantiles=quantiles,
            **weights
        )
        rows.append({
            "seed": int(s),
            "score": stats["score"],
            "rows_cv": stats.get("rows_cv", np.nan),
            "groups_cv": stats.get("groups_cv", np.nan),
            "hist_score": stats.get("hist_score", np.nan),
            "meanstd_score": stats.get("meanstd_score", np.nan),
            "quant_score": stats.get("quant_score", np.nan),
            "row_counts": stats.get("row_counts", None),
            "group_counts": stats.get("group_counts", None),
        })

    results_df = pd.DataFrame(rows).sort_values("score", ascending=True).reset_index(drop=True)
    best_seed = int(results_df.loc[0, "seed"])

    # Optional: preview top seeds
    preview = results_df.head(top_k)[["seed", "score", "rows_cv", "groups_cv", "hist_score", "meanstd_score", "quant_score", "row_counts"]]
    print(preview.to_string(index=False))

    return best_seed, results_df


In [75]:
train_df = pd.read_csv("/home/gavrilev/Thesis/src/datasets/blind_test/processed/cold_rna/outer_fold_1/train.csv")

best_seed, search_df = find_best_seed_for_inner_folds(
    train_df=train_df,
    group_col="Target_RNA_sequence",     # replace
    y_col="pKd",       # replace
    n_splits=5,
    seeds=range(0, 1000),    # try more seeds if desired
)

# Build folds using the best seed
inner_val_dfs = make_inner_group_folds(train_df, group_col="Target_RNA_sequence", n_splits=5, seed=best_seed)


 seed    score  rows_cv  groups_cv  hist_score  meanstd_score  quant_score                row_counts
  203 0.483865 0.240775   0.008475    0.028107       0.223612     0.197880 [238, 357, 200, 272, 189]
  178 0.495604 0.246653   0.008475    0.028231       0.231647     0.201318 [241, 256, 186, 208, 365]
  772 0.505550 0.273292   0.008475    0.028725       0.215880     0.182709 [254, 247, 171, 375, 209]
  212 0.508342 0.247436   0.008475    0.027367       0.244679     0.213925 [255, 262, 225, 353, 161]
  971 0.509480 0.254446   0.008475    0.024985       0.256190     0.195434 [182, 214, 219, 363, 278]
  571 0.515749 0.238194   0.008475    0.024716       0.279430     0.217772 [235, 183, 230, 245, 363]
  348 0.520840 0.302028   0.008475    0.027790       0.188340     0.185230 [206, 185, 211, 395, 259]
  901 0.524674 0.306486   0.008475    0.027595       0.207470     0.165242 [253, 245, 387, 221, 150]
  360 0.524758 0.295117   0.008475    0.025610       0.204461     0.195127 [257, 226, 188, 

In [73]:
inner_val_dfs

[     Entry_ID                                             SMILES  \
 0         270         c1c(ccc2Sc3ccccc3N(c12)CCCNCN(CO)CO)C(=O)C   
 1        1327  C(CN1C=C(CNC2=C3C=CC=CC3=[NH+]C3=C2OC2=C3C=CC=...   
 2        2473  C1C(C(C(C(C1N)OC2C(C(C(C(O2)CN)O)O)N)OC3C(C(C(...   
 3         249  CC1=C(C(=C2C(=C1O)C3(C(=CC(=O)C(C3=O)C(=O)C)O2...   
 4         963             CN1C2=C(N=C1OCCN(C)C)N(C(=O)N(C2=O)C)C   
 ..        ...                                                ...   
 233      2055  NCCCOc1ccc(cc1CNCCC1CCNCC1)c1ccc2c(c1)oc(c2)C(...   
 234       956  N([C@H](C(=O)N1[C@H](CCC1)C(=O)N[C@@H](Cc1cccc...   
 235       292  CN(C)C1C2CC3CC4=C(C=CC(=C4C(=C3C(=O)C2(C(=C(C1...   
 236      2049                NCCCOc1ccc(cc1CNCCCCNC(=N)N)c1cscc1   
 237      2119  CN(C)C1=CC2=C(C=C1)C(=C3C=CC(=[N+](C)C)C=C3O2)...   
 
                                    Target_RNA_sequence  \
 0                        GGCAGAUCUGAGCCUGGGAGCUCUCUGCC   
 1                                           GGGUUAG